[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kithhooni-commits/ds-practice/blob/main/%EC%8B%A4%EC%8A%B55/colab_day3_v2.ipynb)

# 3일차 v2 — 1일차 답 + 2일차 답을 한 모델로

**다른 노트북과 독립적으로 돈다.** 처음부터 끝까지 이것만 실행하면 된다.

## 왜 이 구조인가

전달 곡선을 재보니 3일차가 정확히 분해된다 (val 30장).

| 측정치 영역에서 지운 정도 | 최종 PSNR | 최종 SSIM | 최적 K |
|---|---|---|---|
| 25 dB | 20.24 | 0.5572 | 1e-2 |
| 35 dB | 26.83 | 0.7809 | 3.2e-3 |
| **40 dB** | **29.86** | **0.8626** | 1.8e-3 |
| 45 dB | 32.70 | 0.9066 | 5.6e-4 |
| 완벽 (오차 0) | **71.47** | 0.9999 | 1e-8 |

마지막 줄이 핵심이다. **구조적 한계가 없다.** 측정치에서 노이즈만 완벽히 지우면
2일차 답이 그대로 71 dB 를 낸다. 최종 점수는 오직 디노이징 품질로 결정된다.

노이즈는 흐림 **뒤에** 붙었으므로 측정치 위에서는 백색이다 — **1일차 문제 그대로**다.

    z = net(g)             1일차 답 (DRUNet 37.42 를 warm start)
    x = (D·Z)/(D² + λ)     2일차 답 (역필터). 학습 파라미터는 λ 뿐

손실은 최종 이미지에서 잰다. 그래야 채점 SSIM 을 손실에 넣을 수 있고 λ 도 같이
학습된다 — 위 표에서 최적 K 가 네 자릿수를 움직이므로 고정값으로는 못 맞춘다.

## 통과 기준

**PSNR >= 26, SSIM >= 0.83.** SSIM 이 진짜 관문이다.


## 0. 런타임 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

## 1. 데이터

Drive 에 올린 dataset zip 을 푼다. 이미 풀려 있으면 건너뛴다.

In [ ]:
import os, zipfile, glob
from pathlib import Path

DATA_ROOT = Path("/content/data")
DATA_ROOT.mkdir(exist_ok=True)

if not (DATA_ROOT / "test_deconv_noise").exists():
    zips = sorted(glob.glob("/content/drive/MyDrive/**/dataset*.zip", recursive=True))
    print("찾은 zip:", *zips, sep="\n  ")
    for z in zips[-1:]:
        with zipfile.ZipFile(z) as f:
            f.extractall(DATA_ROOT)
    inner = DATA_ROOT / "dataset"
    if inner.exists() and not (DATA_ROOT / "test_deconv_noise").exists():
        DATA_ROOT = inner

for d in ("train", "val", "test_label", "test_deconv_noise"):
    n = len(glob.glob(str(DATA_ROOT / d / "*.npy")))
    print(f"  {d:<22}{n:>6}장")
assert (DATA_ROOT / "test_deconv_noise" / "noise_meta.json").exists(), "noise_meta.json 이 없다"
print(f"\nDATA_ROOT = {DATA_ROOT}")

## 2. 코드

In [ ]:
REPO = Path("/content/ds-practice")
if REPO.exists():
    !cd "{REPO}" && git pull --ff-only
else:
    !git clone https://github.com/kithhooni-commits/ds-practice.git "{REPO}"

SRC = REPO / "실습5" / "src" / "deconv"
RUNS = Path("/content/runs"); RUNS.mkdir(exist_ok=True)
!cd "{REPO}" && git log --oneline -1

## 3. 1일차 DRUNet 을 찾는다

측정치 영역 디노이저의 출발점으로 쓴다. 없어도 돌아가지만 있으면 훨씬 빨리 수렴한다.
σ 채널은 0 으로 채워 넣으므로 **시작 시점엔 1일차 디노이저와 정확히 같게** 동작한다.

In [ ]:
import torch

def is_day1_denoiser(ck):
    """1일차 train.py 가 저장한 디노이저인가.

    1일차: {"model","layers","features","state_dict","epoch","val_psnr","val_ssim"}
    2/3일차 deconv: 위에 더해 {"input","target","unroll_iters","refine",...}

    "input" 이나 "unroll_iters" 가 있으면 deconv 실행이다. 그 가중치는 이미지 영역
    역산을 배운 것이라 측정치 영역 디노이저 자리에 넣어봐야 도움이 안 된다.
    """
    if "input" in ck or "unroll_iters" in ck or "target" in ck:
        return False
    return ck.get("model") == "drunet" and not ck.get("label_free")

CAND = sorted(set(
    glob.glob("/content/drive/MyDrive/**/*.ckpt", recursive=True)
    + glob.glob("/content/runs/**/*.ckpt", recursive=True)))
D1, best = None, -1
print(f"{chr(32)}{'체크포인트':<57}{'val':>8}  판정")
print("-" * 78)
for c in CAND:
    try:
        ck = torch.load(c, map_location="cpu", weights_only=False)
    except Exception:
        continue
    v = ck.get("val_psnr", 0)
    if is_day1_denoiser(ck):
        verdict = "1일차 디노이저"
        if v > best:
            D1, best = c, v
    elif ck.get("model") == "drunet":
        verdict = f"deconv 실행 (input={ck.get('input')}) — 쓰지 않는다"
    else:
        verdict = f"model={ck.get('model')} — 대상 아님"
    print(f"{c[-57:]:<58}{v:>8.2f}  {verdict}")

INIT = f'--init-refine "{D1}"' if D1 else ""
print()
if D1:
    print(f"선택: {D1}  (val {best:.2f} dB)")
else:
    print("1일차 DRUNet 을 못 찾았다 — 무작위 초기화로 진행한다 (그래도 돌아간다).")
    print("1일차 실행의 checkpoint_best.ckpt 를 Drive 에 올리면 훨씬 빨리 수렴한다.")

## 4. 넘어야 할 선

배포 baseline 과 학습 없는 조합들. K 는 전부 **val 에서** 고른다.

In [ ]:
!cd "{SRC}" && python run_day3.py --data "{DATA_ROOT}" --n-val 60

## (폐기) 2단 분해 — twostage

측정치 영역에서 디노이즈하고 역필터를 거는 구조를 시도했다. **실패했다.**

    ep00 16.10 -> ep04 15.63    loss 는 내려가는데 val PSNR 이 떨어진다

전달 곡선(측정치 영역 정확도 -> 최종) 자체는 맞았지만 **필요한 난이도**를 계산하지
않은 것이 잘못이었다.

| | 입력 | 목표 | 필요 이득 |
|---|---|---|---|
| 1일차 디노이징 (실제 달성) | 25.95 dB | 37.42 dB | +11.5 dB |
| twostage 가 요구하는 것 | 13.50 dB | 40 dB | **+26.5 dB** |

"1일차 문제로 환원된다" 는 맞지만 **1일차보다 훨씬 어려운 1일차 문제**로 환원된다.

  - 역필터가 네트워크의 오차를 1/D 로 증폭한다. end-to-end 는 이미지를 직접
    내놓으므로 그 증폭 자체가 없다
  - 이 데이터는 선명한 주기적 격자라 **이미지 영역**에서 사전지식이 강하다.
    측정치 영역의 `h*f` 는 뭉개져 있어 훨씬 덜 예측 가능하다

**아래 `## 11` 의 전개형을 쓸 것.** 코드는 `twostage.py` 에 남아 있다.


## 6. 평가

`--self-ensemble` 은 4x 다. dipole 이 견디는 대칭만 쓴다 (좌우·상하·180도).
90도 회전은 B0 방향을 돌려 연산자를 바꾸므로 **쓰면 손해**다.

`--sigma-ablation` 은 가중치를 그대로 두고 σ 입력만 바꿔 σ 를 실제로 쓰는지 잰다 —
학습 없는 ablation 이라 대조군을 따로 60 에폭 돌릴 필요가 없다.

In [ ]:
import torch
rows = []
for ck in sorted(RUNS.glob("*/checkpoints/checkpoint_best.ckpt")):
    name = ck.parent.parent.name
    v = torch.load(ck, map_location="cpu", weights_only=False).get("val_psnr", -1)
    rows.append((v, name, ck))
    print(f"{name:<48}{v:>8.2f}")

for v, name, ck in sorted(rows, reverse=True):
    print(f"\n{'='*66}\n  {name}   (val {v:.2f})\n{'='*66}")
    !cd "{SRC}" && python eval_day3.py --data "{DATA_ROOT}" --ckpt "{ck}" --self-ensemble --sigma-ablation

## 7. 융합

구조가 다르면 틀리는 방식도 달라서 평균이 둘 다보다 좋다. 무게는 val 에서 고른다.
SSIM 도 같이 나오니, 융합으로 SSIM 이 떨어지면 쓰지 않으면 된다.

In [ ]:
CKS = [str(c) for _, _, c in sorted(rows, reverse=True)[:3]]
print("섞을 모델:", *[Path(c).parent.parent.name for c in CKS], sep="\n  ")
ARGS = " ".join(f'"{c}"' for c in CKS)
!cd "{SRC}" && python fuse_day3.py --data "{DATA_ROOT}" --ckpts {ARGS} --self-ensemble

## 8. 그림 — 노트북 안에서 바로 본다

배포 안내가 요구한 것: synthetic 학습 쌍과 test 결과 visualize, difference map,
detail zoom-in, 어떤 노이즈·이미지에 취약한지 분석.

In [ ]:
import json
from IPython.display import Image, display, Markdown

BEST = sorted(rows, reverse=True)[0][2]
cfg_p = BEST.parent.parent / "config.json"
cfg = json.loads(cfg_p.read_text(encoding="utf-8")) if cfg_p.exists() else {}
POST = "--post-wiener 0.00562" if cfg.get("target") == "measure" else ""
FIGDIR = Path("/content/figures"); FIGDIR.mkdir(exist_ok=True)

!cd "{SRC}" && python figures_day3.py --data "{DATA_ROOT}" --ckpt "{BEST}" {POST} --self-ensemble --out "{FIGDIR}"

TITLES = {
    "day3_forward_chain": "1. 열화 사슬 — 노이즈가 흐림 뒤에 붙는다",
    "day3_methods_grid":  "2. 노이즈 종류별 x 방법별 복원 결과",
    "day3_diff_zoom":     "3. difference map 과 zoom-in",
    "day3_weakness":      "4. 어떤 노이즈·어떤 σ 에 취약한가",
}
for stem, t in TITLES.items():
    p = FIGDIR / f"{stem}.png"
    if p.exists():
        display(Markdown(f"### {t}")); display(Image(filename=str(p), width=1100))

## 9. Drive 에 저장

체크포인트는 런타임이 끊기면 사라진다. 발표용 그림도 같이 내보낸다.

In [ ]:
import shutil
OUT = Path("/content/drive/MyDrive/ds_day3"); OUT.mkdir(parents=True, exist_ok=True)
for _, name, ck in rows:
    shutil.copy(ck, OUT / f"{name}.ckpt")
    cfg = ck.parent.parent / "config.json"
    if cfg.exists():
        shutil.copy(cfg, OUT / f"{name}_config.json")
for p in FIGDIR.glob("day3_*"):
    shutil.copy(p, OUT / p.name)
print("저장 ->", OUT)
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:<52}{f.stat().st_size/1e6:>8.1f} MB")

## 11. 되돌아간다 — 전개형 + SSIM 손실  ★ 이것을 돌릴 것

twostage 는 실패했다. 전달 곡선 자체는 맞지만 **필요한 디노이징 난이도**를 계산하지
않은 것이 잘못이었다.

| | 입력 | 목표 | 필요 이득 |
|---|---|---|---|
| 1일차 디노이징 (실제 달성) | 25.95 dB | 37.42 dB | +11.5 dB |
| twostage 가 요구하는 것 | 13.50 dB | 40 dB | **+26.5 dB** |

"1일차 문제로 환원된다"는 맞지만 **1일차보다 훨씬 어려운 1일차 문제**로 환원된다.

  - 역필터가 네트워크의 오차를 1/D 로 증폭한다. end-to-end 는 이미지를 직접
    내놓으므로 그 증폭이 없다
  - 이 이미지들은 선명한 주기적 격자라 **이미지 영역**에서 사전지식이 강하다.
    측정치 영역의 h*f 는 뭉개져 있어 훨씬 덜 예측 가능하다

실측: twostage ep00 16.10 -> ep04 15.63 (loss 는 내려가는데 PSNR 이 떨어진다).

### 검증된 경로로 돌아간다

    전개형 (unet f32, blind)     25.91   ep37
    전개형 (DRUNet f48 + σ)      26.85   ep15   <- PSNR 기준 이미 통과

부족했던 것은 SSIM 하나뿐이다 (0.77 vs 기준 0.83). 그것만 얹는다.

In [ ]:
!cd "{SRC}" && python train_deconv.py \
    --model unrolled --refine drunet --features 48 --unroll-iters 4 \
    --sigma-map --share-weights \
    --loss charbonnier_ssim --ssim-weight 0.5 \
    --noise-model challenge --input measure \
    --epochs 60 --batch 4 --lr 2e-4 --clip-grad 1.0 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag unrolled_ssim

### 11b. SSIM 이 그래도 0.83 에 못 미치면

SSIM 항 비중을 0.84 로 (Zhao et al. 권장값). PSNR 을 조금 내주고 SSIM 을 산다.
11 번이 0.83 을 넘겼으면 **돌릴 필요 없다.**

In [ ]:
!cd "{SRC}" && python train_deconv.py \
    --model unrolled --refine drunet --features 48 --unroll-iters 4 \
    --sigma-map --share-weights \
    --loss charbonnier_ssim --ssim-weight 0.84 \
    --noise-model challenge --input measure \
    --epochs 60 --batch 4 --lr 2e-4 --clip-grad 1.0 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag unrolled_ssim084

## 12. SSIM 미세조정  ★ 이것을 돌릴 것

SSIM 을 **처음부터** 손실에 넣으면 해롭다는 것이 실측으로 확인됐다.

    v2 (charbonnier_ssim 0.5, 처음부터)   ep00 18.73/0.6725 -> ep05 17.53/0.6730
                                          PSNR 은 떨어지고 SSIM 은 제자리다

덜 학습된 모델에서 SSIM 은 "정답과 맞든 아니든 국소 대비를 키우는" 쪽으로 민다.
정확도를 벌기 전에 대비부터 흉내내게 되는 것이다.

정공법은 **PSNR 을 먼저 벌고 SSIM 으로 미세조정**하는 것이다. v1 이 이미
27.92 dB / 0.8004 에 도달해 있으니 거기서 이어간다.

  - `--init-model` 로 v1 의 가중치를 이어받는다 (같은 구조여야 한다)
  - `--ssim-weight 0.84` 로 SSIM 을 세게 건다
  - `--lr 5e-5` — 미세조정이므로 학습률을 낮춘다. 크면 벌어둔 PSNR 을 잃는다
  - 15 에폭이면 충분하다 (약 40분)

### 먼저: v1 체크포인트를 Drive 로 꺼낸다

노트북을 다른 런타임에서 돌리면 `/content/runs` 를 공유하지 않는다. **v1 런타임에서** 아래 셀을 돌려야 이쪽에서 이어받을 수 있다.


In [ ]:
# ★ v1 런타임(다른 노트북)에서 이 셀을 돌려 체크포인트를 Drive 로 꺼낼 것.
#   런타임이 끊기면 /content/runs 는 사라진다. 지금 우리 최고 모델이 거기 있다.
import shutil, torch, glob
from pathlib import Path
OUT = Path("/content/drive/MyDrive/ds_day3"); OUT.mkdir(parents=True, exist_ok=True)
for c in sorted(glob.glob("/content/runs/**/checkpoint_best.ckpt", recursive=True)):
    ck = torch.load(c, map_location="cpu", weights_only=False)
    name = Path(c).parent.parent.name
    shutil.copy(c, OUT / f"{name}.ckpt")
    print(f"{name:<48}{ck.get('val_psnr', 0):>8.2f}{ck.get('val_ssim', 0):>9.4f}  -> Drive")


In [ ]:
# 이어받을 체크포인트를 고른다.
#
# 노트북 두 개를 각각 다른 런타임에서 돌리면 /content/runs 를 공유하지 않는다.
# 그래서 Drive 도 같이 뒤지고, glob 순서가 아니라 **val PSNR 로** 고른다.
import torch, glob
from pathlib import Path

CAND = sorted(set(glob.glob(str(RUNS / "**/checkpoint_best.ckpt"), recursive=True)
                  + glob.glob("/content/drive/MyDrive/**/*.ckpt", recursive=True)))
rows = []
for c in CAND:
    try:
        ck = torch.load(c, map_location="cpu", weights_only=False)
    except Exception:
        continue
    if ck.get("model") != "unrolled":
        continue
    rows.append((ck.get("val_psnr", -1), ck.get("val_ssim", 0), c, ck))

if not rows:
    raise SystemExit("전개형 체크포인트를 못 찾았다. v1 런타임에서 Drive 로 복사할 것 "
                     "(아래 '체크포인트를 Drive 로' 셀 참고)")

print(f"{'체크포인트':<52}{'PSNR':>8}{'SSIM':>9}")
print("-" * 70)
for p, s, c, ck in sorted(rows, reverse=True):
    print(f"{c[-51:]:<52}{p:>8.2f}{s:>9.4f}")

BEST = max(rows)
BASE, ck = BEST[2], BEST[3]
print(f"{chr(10)}선택: {BASE}")
print(f"  val {BEST[0]:.2f} dB / {BEST[1]:.4f}   "
      f"(f{ck.get('features')} iters={ck.get('unroll_iters')} sigma_map={ck.get('sigma_map')})")
if BEST[0] < 25:
    print(f"{chr(10)}[주의] val 이 {BEST[0]:.2f} 로 낮다. v1(27.9 대) 체크포인트가 이 런타임에 "
          f"없는 것이다. 미세조정은 좋은 모델에서 출발해야 의미가 있다 — "
          f"v1 런타임에서 Drive 로 복사한 뒤 다시 실행할 것")

In [ ]:
!cd "{SRC}" && python train_deconv.py \
    --model unrolled --refine drunet --features 48 --unroll-iters 4 \
    --sigma-map --share-weights --init-model "{BASE}" \
    --loss charbonnier_ssim --ssim-weight 0.84 \
    --noise-model challenge --input measure \
    --epochs 15 --batch 4 --lr 5e-5 --clip-grad 1.0 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag ssim_finetune

## 13. 2-step 제대로 — 융합 재료용 (여유 있으면)

다른 팀 결과를 보면 2-step 이 **salt & pepper 에서 +2.66 dB** 로 압도적이다.

| noise | 1-step | 2-step | |
|---|---|---|---|
| gaussian | 27.97 | 27.20 | 1-step |
| rician | 22.11 | 22.06 | 둘 다 최약 |
| **salt & pepper** | 28.44 | **31.11** | **2-step +2.66 dB** |
| uniform | 27.50 | 26.56 | 1-step |

임펄스는 **증폭되기 전에 측정치 영역에서** 지워야 한다. 우리도 같은 것을 봤다 —
median → Wiener 가 s&p 에서만 딥러닝을 이겼다 (23.23 vs 19.55).

두 방법이 **상호보완적**이므로 융합 재료로 맞다. 전체 평균으로는 우리 1-step 계열이
앞서므로 이건 보조다.

### 내 첫 twostage 가 실패한 이유

처음부터 최종 이미지 손실로 end-to-end 학습해서 **디노이저가 자기 목표를 직접 배울
기회가 없었다**. 제대로 된 순서는 두 단계다.

    (1) --target measure 로 h*f 를 직접 맞히게 지도학습    <- 잘 조건화된 문제
    (2) Wiener 를 달고 최종 이미지 손실로 joint 미세조정   <- 채점 지표에 정렬

In [ ]:
!cd "{SRC}" && python train_deconv.py \
    --model drunet --features 64 --target measure \
    --noise-model challenge --input measure \
    --epochs 40 --batch 8 --lr 2e-4 --loss charbonnier --clip-grad 1.0 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag stage1_measure

### (2) Wiener 를 달고 joint 미세조정

1단계 가중치를 `--init-refine` 로 이어받는다. λ 초기값은 저쪽이 쓴 K=3e-3 에 맞춘다
(전달 곡선에서 40 dB 디노이징일 때의 최적값과 같은 자리다).

In [ ]:
S1 = sorted(RUNS.glob("*stage1_measure*/checkpoints/checkpoint_best.ckpt"))[-1]
print("1단계:", S1)
!cd "{SRC}" && python train_deconv.py \
    --model twostage --refine drunet --features 64 \
    --sigma-map --init-refine "{S1}" --init-lam 3e-3 \
    --loss charbonnier_ssim --ssim-weight 0.84 \
    --noise-model challenge --input measure \
    --epochs 10 --batch 8 --lr 5e-5 --clip-grad 1.0 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag twostage_joint

## 14. end-to-end DRUNet — 다른 조가 최고라는 그것

용어를 먼저 맞추자. **우리 전개형도 end-to-end 로 학습한다** (측정치 -> 이미지, 손실은
최종 이미지에서). 다른 조가 말하는 "end-to-end" 는 보통 **한 개의 평범한 네트워크**,
즉 저 표의 "1-step" 이다.

    다른 조 1-step (test)   26.378 / 0.8182
    우리 전개형 v1 (val)    27.92  / 0.8004

**val 과 test 를 비교하는 것이라 그대로 믿으면 안 된다.** 우리는 아직 test 숫자가
하나도 없다 — 그것이 지금 가장 큰 구멍이다.

그래도 이 구조는 돌려볼 값이 있다.

  - 전개형보다 **훨씬 싸다** (반복 4회가 없어 batch 8, 에폭당 절반 이하)
  - 구조가 다르므로 **틀리는 방식도 달라서** 융합 재료로 좋다
  - 다른 조에서 잘 나온다면 우리 데이터에서도 확인해 둘 가치가 있다

In [ ]:
!cd "{SRC}" && python train_deconv.py \
    --model drunet --features 64 --input measure --target label \
    --loss charbonnier \
    --noise-model challenge \
    --epochs 60 --batch 8 --lr 2e-4 --clip-grad 1.0 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag e2e_drunet

## 그림과 발표 자료

`python-pptx` 와 한글 폰트를 먼저 깐다. **matplotlib 캐시를 지워야** 새 폰트를
인식한다 — `figures_day3.py` 는 별도 프로세스로 돌기 때문이다. (1일차에 이걸로 한 번
그림이 전부 네모로 나왔다.)

In [ ]:
# 그림·발표 자료 준비 (한 번만)
#   fonts-nanum 을 깐 뒤 matplotlib 캐시를 **지우고 디렉터리를 다시 만든다**.
#   지우기만 하면 캐시를 쓰지 못해 경고가 뜨고, 매번 폰트를 다시 훑는다.
#   figures_day3.py 는 별도 프로세스로 도니 여기서 미리 정리해 둔다.
!pip -q install python-pptx
!apt-get -qq install -y fonts-nanum fonts-nanum-coding > /dev/null 2>&1
!fc-cache -f > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib && mkdir -p ~/.cache/matplotlib

import matplotlib
import matplotlib.font_manager as fm
fm.fontManager.__init__()                      # 폰트 목록을 새로 훑는다
have = sorted({f.name for f in fm.fontManager.ttflist
               if "Nanum" in f.name or "Malgun" in f.name})
print("한글 폰트:", have if have else "없음")
if not have:
    print("  -> 그림의 한글이 네모로 나온다. 런타임 재시작 후 이 셀을 다시 돌릴 것")

from pathlib import Path
REPO = Path("/content/ds-practice")
SRC = REPO / "실습5" / "src" / "deconv"
FIGDIR = Path("/content/figures"); FIGDIR.mkdir(exist_ok=True)
print("SRC =", SRC, "|", "있음" if SRC.exists() else "없음 — 코드 받기 셀을 먼저")

In [ ]:
!cd "{SRC}" && python figures_day3.py --data "{DATA_ROOT}" --ckpt "{CK}" --self-ensemble --out "{FIGDIR}"

from IPython.display import Image, display, Markdown
TITLES = {
    "day3_forward_chain": "1. 열화 사슬 — 노이즈가 흐림 뒤에 붙는다",
    "day3_methods_grid":  "2. 노이즈 종류별 x 방법별 복원 결과",
    "day3_diff_zoom":     "3. difference map 과 zoom-in",
    "day3_weakness":      "4. 어떤 노이즈·어떤 σ 에 취약한가",
}
for stem, t in TITLES.items():
    p = FIGDIR / f"{stem}.png"
    if p.exists():
        display(Markdown(f"### {t}")); display(Image(filename=str(p), width=1100))
    else:
        print("없음:", p)

### 발표 슬라이드 13장

요구사항 1(파이프라인) · 2(before/after/difference/GT) · 3(왜 그 방법인가) ·
4(label-free, 보너스) + 시도별 요약 한 페이지 + test 규칙 슬라이드.

In [ ]:
NAME = "본인이름"          # 발표자 이름을 넣을 것
PSNR, SSIM = 29.25, 0.8777   # eval_day3 가 낸 test 제출값

# 그림은 저장소의 figures/ 를 읽으므로 방금 만든 것을 그리로 옮긴다
import shutil
DEST = REPO / "실습5" / "figures"; DEST.mkdir(parents=True, exist_ok=True)
for p in FIGDIR.glob("day3_*"):
    shutil.copy(p, DEST / p.name)

!cd "{SRC}" && python make_ppt3.py --psnr {PSNR} --ssim {SSIM} --name "{NAME}"

OUT = Path("/content/drive/MyDrive/ds_day3"); OUT.mkdir(parents=True, exist_ok=True)
for p in list(DEST.glob("day3_*")) + list((REPO / "실습5").glob("*.pptx")):
    shutil.copy(p, OUT / p.name)
    print(f"  {p.name:<44}{p.stat().st_size/1e6:>7.2f} MB  -> Drive")

## A. rician 대책 — noise-stats  ★ v1 런타임에서

test 결과에서 rician 만 22.47 로 나머지(29~35)보다 7 dB 뒤진다. 100장 중 25장이라
평균을 크게 끌어내린다 — **rician 을 25 dB 로만 올려도 전체가 +0.63 dB** 오른다.

원인은 밝기 편향이다. val 40장에서 잰 평균 편향:

| | 평균 편향 | 널원뿔 첨도 |
|---|---|---|
| gaussian | +0.00014 | 2.91 |
| **rician** | **+0.0396** | **10.39** |
| uniform | −0.00010 | 3.84 |
| salt & pepper | −0.00388 | 3.38 |

rician 은 정류라 밝기를 위로 민다. dipole 은 DC 를 1/3 로 보존하므로 역산에서 3배가
되어 복원 이미지에 **0.119** 의 오차로 남는다 — 이미지 std 가 0.222 인데 그렇다.
상수 오프셋 0.119 만으로 PSNR 이 18.5 dB 수준이니 22.47 이 설명된다.

**그런데 첨도가 rician 을 깔끔하게 가른다** (10.39 vs 2.9~3.8). σ 하나 대신
`(σ, 왜도, 첨도)` 를 조건으로 주면 네트워크가 종류를 알아본다. 널 원뿔에는 신호가
없으니 거기 남은 값의 모양이 곧 노이즈의 모양이다 — **측정치 하나에서 전부 나온다.**

시작 로그에 이 줄이 떠야 한다:

    noise-stats: 널 원뿔에서 (σ, 왜도, 첨도) 를 읽어 준다 — rician 을 구분하기 위해서

In [ ]:
!cd "{SRC}" && python train_deconv.py \
    --model unrolled --refine drunet --features 48 --unroll-iters 4 \
    --sigma-map --noise-stats --share-weights \
    --noise-model challenge --input measure \
    --epochs 80 --batch 4 --lr 2e-4 --clip-grad 1.0 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag stats3

## B. v1 을 이어서 더 학습  ★ 다른 런타임에서

v1 은 **수렴하지 않았다.** ep51 28.351 -> ep56 28.437 로 끝까지 오르고 있었다.
60 에폭이 모자랐던 것이다. `--init-model` 로 이어받아 코사인을 한 바퀴 더 돈다.
미세조정이므로 `--lr 1e-4` 로 낮춘다.

In [ ]:
CK = "/content/drive/MyDrive/ds_day3/0902-0418_deconv-measure_u_drunet_sig.ckpt"
import os; print("있음" if os.path.exists(CK) else "없음 — Drive 경로를 확인할 것")
!cd "{SRC}" && python train_deconv.py \
    --model unrolled --refine drunet --features 48 --unroll-iters 4 \
    --sigma-map --share-weights --init-model "{CK}" \
    --noise-model challenge --input measure \
    --epochs 60 --batch 4 --lr 1e-4 --clip-grad 1.0 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag v1_more